# Machine Learning Demo with Toy Titanic Dataset

Suggested Reading:
* [Types of Categorical Encoding](https://www.analyticsvidhya.com/blog/2020/08/types-of-categorical-data-encoding/)
* [Pipeline, ColumnTransformer and FeatureUnion explained](https://towardsdatascience.com/pipeline-columntransformer-and-featureunion-explained-f5491f815f)
* [Dealing With Missing Values in Python – A Complete Guide](https://www.analyticsvidhya.com/blog/2021/05/dealing-with-missing-values-in-python-a-complete-guide/)

## Imports / Presets

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
pd.set_option("display.max_rows", 500)

#For dataset
import seaborn as sns

import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer

from category_encoders.target_encoder import TargetEncoder

## Custom Classes/Functions

In [2]:
class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, col_names=None, col_nums=None):
        self.col_names=col_names
        self.col_nums=col_nums
        self.use = None
        assert (self.col_names is not None) or (self.col_nums is not None), 'Must set either col_names or col_nums'
        
    def fit(self, X, y=None):
        if self.col_names is not None:
            self.use = 'col_names'
        elif self.col_nums is not None:
            self.use = 'col_nums'
        return self
    
    def transform(self, X, y=None):
        if self.use == 'col_names':
            _X = X[self.col_names].copy()
        elif self.use == 'col_nums':
            _X = X[:, self.col_nums]
        return(_X)
        

In [3]:
def encode_dataset(X_train, y_train, X_test, cat_columns, encoder):
    noncat_train = X_train.drop(columns=cat_columns)
    noncat_test = X_test.drop(columns=cat_columns)
    cat_train = X_train[cat_columns]
    cat_test = X_test[cat_columns]
    
    cat_train_encoded = encoder.fit_transform(cat_train, y_train)
    cat_test_encoded = encoder.transform(cat_test)
    
    X_train_encoded = noncat_train.join(cat_train_encoded)
    X_test_encoded = noncat_test.join(cat_test_encoded)
    return(X_train_encoded, X_test_encoded)

## Data

In [4]:
df = sns.load_dataset('titanic')

## Preprocessing

### Encoding Binary Variables

In [5]:
encoder = LabelEncoder()
df['male'] = encoder.fit_transform(y=df['sex'])

df['alone'] = df['alone'].astype(int)

df.drop(columns=['sex','who','adult_male','alive', 'embarked','pclass'], inplace=True)

In [6]:
df.head()

,survived,age,sibsp,parch,fare,class,deck,embark_town,alone,male
0,0,22.0,1,0,7.2500,Third,NaN,Southampton,0,1
1,1,38.0,1,0,71.2833,First,C,Cherbourg,0,0
2,1,26.0,0,0,7.9250,Third,NaN,Southampton,1,0
3,1,35.0,1,0,53.1000,First,C,Southampton,0,0
4,0,35.0,0,0,8.0500,Third,NaN,Southampton,1,1


### Splitting (Train/Val/Test)

In [7]:
df.shape

(891, 10)

In [8]:
X = df.drop(columns='survived')
y = df.survived

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## Baseline Model

Simplest possible model: Drop all categoricals, fill nas with 0.

In [10]:
categorical_columns = ['class', 'deck', 'embark_town']

In [11]:
X_train_baseline = X_train.drop(columns=categorical_columns).fillna(0)
X_test_baseline = X_test.drop(columns=categorical_columns).fillna(0)

In [12]:
model = RandomForestClassifier(random_state=42)

In [13]:
model.fit(X_train_baseline, y_train)

C:\Users\User\Anaconda3\lib\site-packages\sklearn\ensemble\forest.py:245: FutureWarning: The default value of n_estimators will change from 10 in version 0.20 to 100 in 0.22.
  "10 in version 0.20 to 100 in 0.22.", FutureWarning)


RandomForestClassifier(bootstrap=True, class_weight=None, criterion='gini',
                       max_depth=None, max_features='auto', max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=10,
                       n_jobs=None, oob_score=False, random_state=42, verbose=0,
                       warm_start=False)

#### With .predict()...

In [14]:
y_preds = model.predict(X_test_baseline)

In [15]:
y_preds[0:10]

array([0, 0, 0, 1, 0, 1, 1, 0, 1, 1], dtype=int64)

In [16]:
roc_auc_score(y_test, y_preds)

0.7587623679356028

#### With .predict_proba()

In [17]:
y_preds_proba = model.predict_proba(X_test_baseline)[:,1]

In [18]:
y_preds_proba[0:10]

array([0.3       , 0.        , 0.        , 1.        , 0.4       ,
       1.        , 0.88161616, 0.2       , 0.7       , 1.        ])

In [19]:
roc_auc_score(y_test, y_preds_proba)

0.8367013248364916

## Model Improvement

The baseline score is 0.836. We want to find ways to improve this.

### Baseline Model w/ Pipeline

In [20]:
categorical_columns = ['class', 'deck', 'embark_town']
numeric_columns = [c for c in X_train.columns if c not in categorical_columns]

In [21]:
pipe = Pipeline([
    ('column_selector', ColumnSelector(col_names = numeric_columns)),
    ('imputer', SimpleImputer(strategy='constant',fill_value=0)),
    ('random_forest', RandomForestClassifier(random_state=42)),
])

In [22]:
pipe.fit(X_train, y_train)

C:\Users\User\Anaconda3\lib\site-packages\sklearn\ensemble\forest.py:245: FutureWarning: The default value of n_estimators will change from 10 in version 0.20 to 100 in 0.22.
  "10 in version 0.20 to 100 in 0.22.", FutureWarning)


Pipeline(memory=None,
         steps=[('column_selector',
                 ColumnSelector(col_names=['age', 'sibsp', 'parch', 'fare',
                                           'alone', 'male'],
                                col_nums=None)),
                ('imputer',
                 SimpleImputer(add_indicator=False, copy=True, fill_value=0,
                               missing_values=nan, strategy='constant',
                               verbose=0)),
                ('random_forest',
                 RandomForestClassifier(bootstrap=True, class_weight=None,
                                        criterion='gini', max_depth=None,
                                        max_features='auto',
                                        max_leaf_nodes=None,
                                        min_impurity_decrease=0.0,
                                        min_impurity_split=None,
                                        min_samples_leaf=1, min_samples_split=2,
                 

In [23]:
y_preds_proba = pipe.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_preds_proba)

0.8367013248364916

### Target Encoding

In [24]:
categorical_columns = ['class', 'deck', 'embark_town']
numeric_columns = [c for c in X_train.columns if c not in categorical_columns]

In [25]:
X_train_te, X_test_te = encode_dataset(X_train, y_train, X_test, categorical_columns, TargetEncoder())

C:\Users\User\Anaconda3\lib\site-packages\category_encoders\utils.py:21: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
  elif pd.api.types.is_categorical(cols):


In [26]:
X_train.head()

,age,sibsp,parch,fare,class,deck,embark_town,alone,male
298,NaN,0,0,30.5000,First,C,Southampton,1,1
884,25.00,0,0,7.0500,Third,NaN,Southampton,1,1
247,24.00,0,2,14.5000,Second,NaN,Southampton,0,0
478,22.00,0,0,7.5208,Third,NaN,Southampton,1,1
305,0.92,1,2,151.5500,First,C,Southampton,0,1


In [27]:
X_train_te.head()

,age,sibsp,parch,fare,alone,male,class,deck,embark_town
298,NaN,0,0,30.5000,1,1,0.611842,0.586957,0.336049
884,25.00,0,0,7.0500,1,1,0.245333,0.300000,0.336049
247,24.00,0,2,14.5000,0,0,0.482270,0.300000,0.336049
478,22.00,0,0,7.5208,1,1,0.245333,0.300000,0.336049
305,0.92,1,2,151.5500,0,1,0.611842,0.586957,0.336049


In [28]:
pipe = Pipeline([
    ('column_selector', ColumnSelector(col_names = numeric_columns + categorical_columns)),
    ('imputer', SimpleImputer(strategy='constant',fill_value=0)),
    ('random_forest', RandomForestClassifier(random_state=42)),
])

In [29]:
pipe.fit(X_train_te, y_train)

C:\Users\User\Anaconda3\lib\site-packages\sklearn\ensemble\forest.py:245: FutureWarning: The default value of n_estimators will change from 10 in version 0.20 to 100 in 0.22.
  "10 in version 0.20 to 100 in 0.22.", FutureWarning)


Pipeline(memory=None,
         steps=[('column_selector',
                 ColumnSelector(col_names=['age', 'sibsp', 'parch', 'fare',
                                           'alone', 'male', 'class', 'deck',
                                           'embark_town'],
                                col_nums=None)),
                ('imputer',
                 SimpleImputer(add_indicator=False, copy=True, fill_value=0,
                               missing_values=nan, strategy='constant',
                               verbose=0)),
                ('random_forest',
                 RandomForestClassifier(bootstrap=True, class_weight=None,
                                        criterion='gini', max_depth=None,
                                        max_features='auto',
                                        max_leaf_nodes=None,
                                        min_impurity_decrease=0.0,
                                        min_impurity_split=None,
                       

In [30]:
y_preds_proba = pipe.predict_proba(X_test_te)[:,1]
roc_auc_score(y_test, y_preds_proba)

0.8596344122086198

### With ColumnTransformer

In [31]:
categorical_columns = ['class', 'deck', 'embark_town']
numeric_columns = [c for c in X_train.columns if c not in categorical_columns]

In [32]:
# Define categorical pipeline
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', TargetEncoder())
])

# Define numerical pipeline
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant',fill_value=0)),
])

# Combine categorical and numerical pipelines
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, categorical_columns),
    ('num', num_pipe, numeric_columns)
])

# Fit a pipeline with transformers and an estimator to the training data
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])
pipe.fit(X_train, y_train)

C:\Users\User\Anaconda3\lib\site-packages\category_encoders\utils.py:21: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
  elif pd.api.types.is_categorical(cols):
C:\Users\User\Anaconda3\lib\site-packages\sklearn\ensemble\forest.py:245: FutureWarning: The default value of n_estimators will change from 10 in version 0.20 to 100 in 0.22.
  "10 in version 0.20 to 100 in 0.22.", FutureWarning)


Pipeline(memory=None,
         steps=[('preprocessor',
                 ColumnTransformer(n_jobs=None, remainder='drop',
                                   sparse_threshold=0.3,
                                   transformer_weights=None,
                                   transformers=[('cat',
                                                  Pipeline(memory=None,
                                                           steps=[('imputer',
                                                                   SimpleImputer(add_indicator=False,
                                                                                 copy=True,
                                                                                 fill_value='missing',
                                                                                 missing_values=nan,
                                                                                 strategy='constant',
                                                      

In [33]:
y_preds_proba = pipe.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_preds_proba)

0.8538906590642293

### One Hot Encoding

In [34]:
categorical_columns = ['class', 'deck', 'embark_town']
numeric_columns = [c for c in X_train.columns if c not in categorical_columns]

In [35]:
# Define categorical pipeline
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

# Define numerical pipeline
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant',fill_value=0)),
])

# Combine categorical and numerical pipelines
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, categorical_columns),
    ('num', num_pipe, numeric_columns)
])

# Fit a pipeline with transformers and an estimator to the training data
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])
pipe.fit(X_train, y_train)

C:\Users\User\Anaconda3\lib\site-packages\sklearn\ensemble\forest.py:245: FutureWarning: The default value of n_estimators will change from 10 in version 0.20 to 100 in 0.22.
  "10 in version 0.20 to 100 in 0.22.", FutureWarning)


Pipeline(memory=None,
         steps=[('preprocessor',
                 ColumnTransformer(n_jobs=None, remainder='drop',
                                   sparse_threshold=0.3,
                                   transformer_weights=None,
                                   transformers=[('cat',
                                                  Pipeline(memory=None,
                                                           steps=[('imputer',
                                                                   SimpleImputer(add_indicator=False,
                                                                                 copy=True,
                                                                                 fill_value='missing',
                                                                                 missing_values=nan,
                                                                                 strategy='constant',
                                                      

In [36]:
y_preds_proba = pipe.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_preds_proba)

0.8382106322320979

## Missing Values

#### Impute -1

In [37]:
categorical_columns = ['class', 'deck', 'embark_town']
numeric_columns = [c for c in X_train.columns if c not in categorical_columns]

In [38]:
# Define categorical pipeline
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', TargetEncoder())
])

# Define numerical pipeline
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'constant', fill_value=-1)),
])

# Combine categorical and numerical pipelines
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, categorical_columns),
    ('num', num_pipe, numeric_columns)
])

# Fit a pipeline with transformers and an estimator to the training data
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])
pipe.fit(X_train, y_train)

C:\Users\User\Anaconda3\lib\site-packages\category_encoders\utils.py:21: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
  elif pd.api.types.is_categorical(cols):
C:\Users\User\Anaconda3\lib\site-packages\sklearn\ensemble\forest.py:245: FutureWarning: The default value of n_estimators will change from 10 in version 0.20 to 100 in 0.22.
  "10 in version 0.20 to 100 in 0.22.", FutureWarning)


Pipeline(memory=None,
         steps=[('preprocessor',
                 ColumnTransformer(n_jobs=None, remainder='drop',
                                   sparse_threshold=0.3,
                                   transformer_weights=None,
                                   transformers=[('cat',
                                                  Pipeline(memory=None,
                                                           steps=[('imputer',
                                                                   SimpleImputer(add_indicator=False,
                                                                                 copy=True,
                                                                                 fill_value='missing',
                                                                                 missing_values=nan,
                                                                                 strategy='constant',
                                                      

In [39]:
y_preds_proba = pipe.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_preds_proba)

0.8539745094750963

### Impute mean

In [40]:
categorical_columns = ['class', 'deck', 'embark_town']
numeric_columns = [c for c in X_train.columns if c not in categorical_columns]

In [41]:
# Define categorical pipeline
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', TargetEncoder())
])

# Define numerical pipeline
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'mean')),
])

# Combine categorical and numerical pipelines
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, categorical_columns),
    ('num', num_pipe, numeric_columns)
])

# Fit a pipeline with transformers and an estimator to the training data
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])
pipe.fit(X_train, y_train)

C:\Users\User\Anaconda3\lib\site-packages\category_encoders\utils.py:21: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
  elif pd.api.types.is_categorical(cols):
C:\Users\User\Anaconda3\lib\site-packages\sklearn\ensemble\forest.py:245: FutureWarning: The default value of n_estimators will change from 10 in version 0.20 to 100 in 0.22.
  "10 in version 0.20 to 100 in 0.22.", FutureWarning)


Pipeline(memory=None,
         steps=[('preprocessor',
                 ColumnTransformer(n_jobs=None, remainder='drop',
                                   sparse_threshold=0.3,
                                   transformer_weights=None,
                                   transformers=[('cat',
                                                  Pipeline(memory=None,
                                                           steps=[('imputer',
                                                                   SimpleImputer(add_indicator=False,
                                                                                 copy=True,
                                                                                 fill_value='missing',
                                                                                 missing_values=nan,
                                                                                 strategy='constant',
                                                      

In [42]:
y_preds_proba = pipe.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_preds_proba)

0.8628207278215663

### Impute median

In [43]:
categorical_columns = ['class', 'deck', 'embark_town']
numeric_columns = [c for c in X_train.columns if c not in categorical_columns]

In [44]:
# Define categorical pipeline
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', TargetEncoder())
])

# Define numerical pipeline
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),
])

# Combine categorical and numerical pipelines
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, categorical_columns),
    ('num', num_pipe, numeric_columns)
])

# Fit a pipeline with transformers and an estimator to the training data
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])
pipe.fit(X_train, y_train)

C:\Users\User\Anaconda3\lib\site-packages\category_encoders\utils.py:21: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
  elif pd.api.types.is_categorical(cols):
C:\Users\User\Anaconda3\lib\site-packages\sklearn\ensemble\forest.py:245: FutureWarning: The default value of n_estimators will change from 10 in version 0.20 to 100 in 0.22.
  "10 in version 0.20 to 100 in 0.22.", FutureWarning)


Pipeline(memory=None,
         steps=[('preprocessor',
                 ColumnTransformer(n_jobs=None, remainder='drop',
                                   sparse_threshold=0.3,
                                   transformer_weights=None,
                                   transformers=[('cat',
                                                  Pipeline(memory=None,
                                                           steps=[('imputer',
                                                                   SimpleImputer(add_indicator=False,
                                                                                 copy=True,
                                                                                 fill_value='missing',
                                                                                 missing_values=nan,
                                                                                 strategy='constant',
                                                      

In [45]:
y_preds_proba = pipe.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_preds_proba)

0.8469310749622673

# Hyperparameter Tuning

So far, the best model uses mean imputation, target encoded categoricals, and a RandomForestClassifier. Let's see if we can improve the score by tuning the hyper parameters.

### With RandomizedSearchCV

In [46]:
categorical_columns = ['class', 'deck', 'embark_town']
numeric_columns = [c for c in X_train.columns if c not in categorical_columns]

In [56]:
# Define categorical pipeline
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', TargetEncoder())
])

# Define numerical pipeline
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'mean')),
])

# Combine categorical and numerical pipelines
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, categorical_columns),
    ('num', num_pipe, numeric_columns)
])

# Fit a pipeline with transformers and an estimator to the training data
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('random_forest', RandomForestClassifier(random_state=42))
])

In [51]:
min_samples_split = [int(x) for x in np.linspace(2,20, num=10)]
min_samples_leaf = [int(x) for x in np.linspace(1,10, num=6)]
max_depth = [int(x) for x in np.linspace(10,110, num=11)]
max_depth.append(None)
n_estimators = [int(x) for x in np.linspace(10,80, num=8)]

param_grid = {
    'random_forest__n_estimators' : n_estimators,
    'random_forest__max_depth' : max_depth,
    'random_forest__min_samples_split' : min_samples_split,
    'random_forest__min_samples_leaf' : min_samples_leaf,
}

In [52]:
param_grid

{'random_forest__n_estimators': [10, 20, 30, 40, 50, 60, 70, 80],
 'random_forest__max_depth': [10,
  20,
  30,
  40,
  50,
  60,
  70,
  80,
  90,
  100,
  110,
  None],
 'random_forest__min_samples_split': [2, 4, 6, 8, 10, 12, 14, 16, 18, 20],
 'random_forest__min_samples_leaf': [1, 2, 4, 6, 8, 10]}

In [59]:
%%time
rs = RandomizedSearchCV(pipe, param_grid, n_iter=10, random_state=42)
rs.fit(X_train, y_train)

Wall time: 3.4 s


In [60]:
y_preds_proba = rs.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_preds_proba)

0.8833640784839847

### With GridSearchCV

In [61]:
categorical_columns = ['class', 'deck', 'embark_town']
numeric_columns = [c for c in X_train.columns if c not in categorical_columns]

In [62]:
# Define categorical pipeline
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', TargetEncoder())
])

# Define numerical pipeline
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy = 'mean')),
])

# Combine categorical and numerical pipelines
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, categorical_columns),
    ('num', num_pipe, numeric_columns)
])

# Fit a pipeline with transformers and an estimator to the training data
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('random_forest', RandomForestClassifier(random_state=42))
])

In [63]:
min_samples_split = [int(x) for x in np.linspace(2,20, num=10)]
min_samples_leaf = [int(x) for x in np.linspace(1,10, num=6)]
max_depth = [int(x) for x in np.linspace(10,110, num=11)]
max_depth.append(None)
n_estimators = [int(x) for x in np.linspace(10,80, num=8)]

param_grid = {
    'random_forest__n_estimators' : n_estimators,
    'random_forest__max_depth' : max_depth,
    'random_forest__min_samples_split' : min_samples_split,
    'random_forest__min_samples_leaf' : min_samples_leaf
}

In [64]:
param_grid

{'random_forest__n_estimators': [10, 20, 30, 40, 50, 60, 70, 80],
 'random_forest__max_depth': [10,
  20,
  30,
  40,
  50,
  60,
  70,
  80,
  90,
  100,
  110,
  None],
 'random_forest__min_samples_split': [2, 4, 6, 8, 10, 12, 14, 16, 18, 20],
 'random_forest__min_samples_leaf': [1, 2, 4, 6, 8, 10]}

In [66]:
%%time
gs = GridSearchCV(pipe, param_grid)
gs.fit(X_train, y_train)

Wall time: 28min 45s


In [67]:
y_preds_proba = gs.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_preds_proba)

0.8846218346469897

# Conclusion

Best model used impute mean, target encoded categoricals, and RandomForestClassifier.